# Qwen3.8 Flash-Next → existing DeepSeek Harness Qwen slot

This is the **drop-in replacement** notebook for the old `Qwen3_8_27B_API_Colab.ipynb`.

Your Windows/Harness side stays unchanged:

- Base URL: `http://127.0.0.1:8787/v1`
- Harness model ID: `qwen3.8-27b`
- Supabase relay ID: `qwen3-8-27b`
- Same `Start-QwenHarnessBridge.ps1`
- Same Harness provider/settings/affinity configuration

Only the Colab backend changes to **Qwen/Qwen3.8-Flash-Next-FP8** using the FreeToken-style A100 + host-RAM runtime. Do not run the old Qwen3.8-27B worker at the same time because both intentionally use the same relay slot.

In Colab **Secrets**, keep the same `QWEN_RELAY_SECRET`. If you already use automatic Oracle wake, keep the same `ORACLE_WAKE_GITHUB_TOKEN`. `HF_TOKEN` is optional.

## Section 1 — Install / update Flash worker
Run once in a fresh Colab runtime. The Flash production path needs an **SM80+ GPU**, about **220+ GiB host RAM**, and roughly **190 GiB free local storage** for the real FP8 checkpoint.

In [ ]:
!rm -rf /content/All-testing
!git clone --depth 1 --branch qwen38-flash-freetoken-colab https://github.com/Logan17de/All-testing.git /content/All-testing
%cd /content/All-testing/llm
%pip install -q -U -r qwen38_flash_freetoken/requirements-colab.txt
%pip install -q -U -e ".[qwen38-flash-freetoken]"
import importlib.metadata as metadata
print(f"all-testing-llm {metadata.version('all-testing-llm')}: OK ✅")

## Section 2A — EXPECTED-OOM real production-path smoke test
This section is intentionally designed to **end in CUDA/VRAM OOM**. That OOM is the PASS condition.

Before it is allowed to pass, it uses the **real Qwen3.8-Flash-Next-FP8 checkpoint** and validates the same important production layers: SM80+/RAM/disk, Supabase/Harness relay preflight, live checkpoint architecture, custom FreeToken expert backend registration, real model loading, host-resident expert banks, GPU expert cache/runtime binding, measured bandwidth, and a real routed text generation.

If the GPU reaches a natural memory limit during the real runtime, that memory failure is accepted. If the model and generation succeed, the test deliberately requests more VRAM than remains so it still terminates at the expected OOM wall. **Any non-memory exception before OOM is a real bug/config/dependency issue and is re-raised.**

This is the Flash equivalent of the old Qwen27B expected-OOM validation; it is not a fit test. The checkpoint cache is reused by Section 2B, so the large model should not need to download again.

In [ ]:
import qwen3_8_flash_supabase_colab_oom_smoke as qwen_oom_smoke
qwen_oom_smoke.main()

## Section 2B — REAL Qwen3.8 Flash Harness worker
Run this after Section 2A reaches `EXPECTED VRAM LIMIT REACHED ✅ — SMOKE TEST PASSED`. It loads Qwen3.8-Flash-Next-FP8, starts the private localhost OpenAI-compatible server inside Colab, then attaches it to the **same outbound Supabase relay used by the old Qwen3.8-27B worker**.

Once it reports that the worker is ready, start/keep your existing Windows `Start-QwenHarnessBridge.ps1` process and use the existing Qwen model in Harness. No local config change is required.

In [ ]:
import qwen3_8_flash_colab_runtime as qwen_worker
qwen_worker.main()

## Section 3 — TESTING: existing Harness relay only (NO GPU/model)
Use a CPU Colab runtime if you only want to prove the existing Windows Harness → Supabase → Colab path. Run Section 1 and then this cell instead of Sections 2A/2B. Every Harness request returns `succeed`.

In [ ]:
import qwen_supabase_test_worker as relay_test
relay_test.main()

## Normal startup

1. Stop/disconnect the old Qwen3.8-27B Colab worker if it is running.
2. Open this notebook with the Flash-capable high-RAM runtime.
3. Run Section 1.
4. Run Section 2A and fix **every non-memory error** until it ends with the expected-OOM PASS banner.
5. Run Section 2B and leave it running.
6. On Windows, keep using the same `Start-QwenHarnessBridge.ps1` and the same `qwen3.8-27b` model entry in Harness.

The model name in Harness intentionally remains `qwen3.8-27b`; underneath, the worker is Qwen3.8 Flash-Next.